BOX-BOX: F1 Strategy Intelligence Dashboard

Phase 5: Stratey Optimization Engine

Simulates every realistic 1-stop and 2-stop strategy per circuit using Phase 3 degradation rates and Phase 4 hazard data, and outputs Plan A / B / C strategy recommendations.

In [1]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import joblib

warnings.filterwarnings('ignore')

In [2]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'data', 'processed'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'), exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase1_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

CONFIGURATION

In [3]:
MIN_STINT_LENGTH = 5
EXTRAPOLATION_BUFFER = 5
TWO_STOP_STEP = 2
DEFAULT_PIT_LOSS = 22.0

LOADING ALL PHASE 3/4 OUTPUTS

In [4]:
def load_all_data():
    log.info("Loading Phase 3 & 4 outputs...")

    degradation_df = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'degradation_profiles.csv'))

    feature_store = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'feature_store.csv'))

    hazard_df = pd.read_csv(os.path.join(BASE, 'data', 'outputs', 'cox_hazard_ratios.csv'))

    engineered_laps = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'engineered_laps.csv'))

    log.info(f" degradation_profiles: {len(degradation_df)} rows")
    log.info(f" feature_store: {len(feature_store)} rows")
    log.info(f" cox_hazard_ratios: {len(hazard_df)} rows")
    log.info(f"engineered_laps: {len(engineered_laps)} rows")

    return degradation_df, feature_store, hazard_df, engineered_laps

BUILD LOOKUP DICTIONARIES

Converts DataFrames into fast dictionary lookups so the simulation loops below don't repeatedly filter DataFrames

In [5]:
def build_lookups(degradation_df, feature_store, hazard_df, engineered_laps):
    log.info("BUILDING LOOKUP STRCUTURES...")

    #degradation_profiles -> {circuit: {compound: {rate, avg_lap_time, max_tyre_life}}}
    deg_lookup = {}
    for _, row in degradation_df.iterrows():
        circuit = row['CircuitName']
        compound = row['Compound']
        deg_lookup.setdefault(circuit, {})[compound] = {
            'degradation_rate': row['DegradationRate'],
            'avg_lap_time': row['AvgLapTime'],
            'max_tyre_life': row['AvgTyreLife'] * 2 #Roughly safety ceiling
        }
    # feature_store -> {circuit: {pit_loss, sc_probability, total_laps}}
    circuit_info = {}
    for _, row in feature_store.iterrows():
        circuit_info[row['CircuitName']] = {
    'pit_loss': row['AvgPitLoss'] if pd.notna(row['AvgPitLoss']) else DEFAULT_PIT_LOSS,
    'sc_probability': row['SafetyCarProbability'],
    'total_laps': int(row['TotalRaceLaps'])
    }

    # cox_hazard -> {circuit: {covariate: hazard_ratio}}
    hazard_lookup = {}
    for _, row in hazard_df.iterrows():
        hazard_lookup.setdefault(row['CircuitName'], {})[row['Covariate']] = row['HazardRatio']

    log.info(f" Circuits with degradation data: {len(deg_lookup)}")
    log.info(f" Circuits with circuit info: {len(circuit_info)}")
    log.info(f" Circuits with hazard data: {len(hazard_lookup)}")

    return deg_lookup, circuit_info, hazard_lookup

STINT TIME CALCULATION

Uses the arithmetic series formula to sum oredicted lap times across a stint without looping lap by lap

In [6]:
def stint_time(avg_lap_time, degradation_rate, stint_length):
    total = stint_length * avg_lap_time
    total += degradation_rate * (stint_length * (stint_length + 1) / 2)
    return total

def is_valid_stint(profile, stint_length):
    if stint_length < MIN_STINT_LENGTH:
        return False
    max_allowed = profile['max_tyre_life'] + EXTRAPOLATION_BUFFER
    if stint_length > max_allowed:
        return False
    return True

RISK SCORE CALCULATION

Uses the Cox Hazard Ratio for CompoundEncoder (a proxy for how much compound choice affects failure risk at this circuit) combined with how far eachs tint pushes the compound's typical stint length, to produce a 0-100 risk score

Higher score = higher strategic risk

In [7]:
def calculate_risk_score(circuit, strategy, deg_profiles, hazard_lookup, sc_probability):
    hazard_info = hazard_lookup.get(circuit, {})
    compound_hazard = hazard_info.get('CompoundEncoded', 1.0)

    overrun_penalty = 0
    for compound, stint_length in strategy:
        profile = deg_profiles.get(compound)
        if profile is None:
            continue
        typical_max = profile['max_tyre_life']
        if stint_length > typical_max:
            overrun_penalty += (stint_length - typical_max) * 2

    '''Base risk from safety car probability (more SC = more
    strategic uncertainity regardless of tire choice)'''
    sc_risk = sc_probability * 40

    #Hazard ratio contribution - scaled to a reasonable range
    hazard_risk = min(abs(compound_hazard - 1.0) * 20, 30)

    total_risk = sc_risk + hazard_risk + overrun_penalty
    return round(min(total_risk, 100), 1)

STRATEGY SIMULATION

In [8]:
def simulate_strategy(circuit, strategy, deg_profiles, pit_loss):
    total_time = 0
    num_stops = len(strategy) - 1

    for compound, stint_length in strategy:
        profile = deg_profiles.get(compound)
        if profile is None: 
            return None
        if not is_valid_stint(profile, stint_length):
            return None
        total_time += stint_time(
            profile['avg_lap_time'], profile['degradation_rate'], stint_length
        )
    total_time += num_stops * pit_loss
    return total_time

FIND ALL VALID STRATEGIES FOR A CIRCUIT

In [9]:
def generate_strategies(circuit, deg_lookup, circuit_info):
    if circuit not in circuit_info or circuit not in deg_lookup:
        return [], [], None

    total_laps = circuit_info[circuit]['total_laps']
    pit_loss = circuit_info[circuit]['pit_loss']
    deg_profiles = deg_lookup[circuit]
    available_compounds = list(deg_profiles.keys())

    one_stop_results = []
    two_stop_results = []

    #1-STOP
    for s1 in range(MIN_STINT_LENGTH, total_laps - MIN_STINT_LENGTH + 1):
        s2 = total_laps - s1
        for c1 in available_compounds:
            for c2 in available_compounds:
                if c1 == c2:
                    continue
                strategy = [(c1,s1), (c2,s2)]
                total_time = simulate_strategy(circuit, strategy, deg_profiles, pit_loss)
                if total_time is not None:
                    one_stop_results.append({'strategy': strategy, 'total_time': total_time})

    #2-STOP
    for s1 in range(MIN_STINT_LENGTH, total_laps - 2 * MIN_STINT_LENGTH + 1, TWO_STOP_STEP):
        for s2 in range(MIN_STINT_LENGTH, total_laps - s1 - MIN_STINT_LENGTH + 1, TWO_STOP_STEP):
            s3 = total_laps - s1 - s2
            if s3 < MIN_STINT_LENGTH:
                continue
            for c1 in available_compounds:
                for c2 in available_compounds:
                    for c3 in available_compounds:
                        if len({c1, c2, c3}) < 2:
                            continue
                        strategy = [(c1, s1), (c2,s2), (c3,s3)]
                        total_time = simulate_strategy(circuit, strategy, deg_profiles, pit_loss)
                        if total_time is not None:
                            two_stop_results.append({'strategy': strategy, 'total_time': total_time})

    return one_stop_results, two_stop_results, deg_profiles

FIND REAL-WORLD MOST COMMON STRATEGY(for Plan B)

Looks at the actual 2024 stint data to find what stop-count and rough compounds sequence was commonly used at this circuit

In [10]:
def find_real_world_strategy(circuit, engineered_laps):
    circuit_laps = engineered_laps[engineered_laps['CircuitName'] == circuit]

    if circuit_laps.empty:
        return None

    #Count stops per driver (max stint number - 1)
    stops_per_driver = circuit_laps.groupby('Driver')['Stint'].max() - 1
    most_common_stops = stops_per_driver.mode()

    if most_common_stops.empty:
        return None

    common_stop_count = int(most_common_stops.iloc[0])

    #Find the most common compound sequence among drivers with that stop count
    drivers_with_common_stops = stops_per_driver[
        stops_per_driver == common_stop_count
    ].index

    sequences = []
    for driver in drivers_with_common_stops:
        driver_laps = circuit_laps[circuit_laps['Driver'] == driver]
        seq = driver_laps.sort_values('LapNumber').groupby('Stint')['Compound'].first().tolist()
        if len(seq) == common_stop_count + 1:
            sequences.append(tuple(seq))

    if not sequences:
        return None

    seq_series = pd.Series(sequences)
    most_common_seq = seq_series.mode()

    if most_common_seq.empty:
        return None

    return {
        'stop_count': common_stop_count,
        'compound_sequence': list(most_common_seq.iloc[0])
    }

BUILD PLAN A/B/C FOR ONE CIRCUIT

In [11]:
def build_plans(circuit, deg_look, circuit_info, hazard_lookup, engineered_laps):
    one_stop, two_stop, deg_profiles = generate_strategies(circuit, deg_look, circuit_info)

    if not one_stop and not two_stop:
        return None

    sc_probability = circuit_info[circuit]['sc_probability']

    #PLAN-A: Mathematically optimal overall
    all_results = one_stop + two_stop
    plan_a = min(all_results, key=lambda x: x['total_time'])

    #PLAN-C: best of the opposite stop count
    plan_a_stops = len(plan_a['strategy']) - 1
    if plan_a_stops == 1 and two_stop:
        plan_c = min(two_stop, key=lambda x: x['total_time'])
    elif plan_a_stops == 2 and one_stop:
        plan_c = min(one_stop, key=lambda x: x['total_time'])
    else:
        plan_c = None

    #PLAN-B: Most common real-world strategy
    real_world = find_real_world_strategy(circuit, engineered_laps)
    plan_b = None

    if real_world is not None:
        total_laps = circuit_info[circuit]['total_laps']
        n_stints = real_world['stop_count'] + 1
        avg_stint_len = total_laps // n_stints

        '''Approximate stint lengths evenly since we only
        know the compound sequence, not exact real lap splits'''
        approx_strategy = []
        remaining = total_laps
        for i, compound in enumerate(real_world['compound_sequence']):
            if i == len(real_world['compound_sequence']) - 1:
                length = remaining
            else:
                length = avg_stint_len
                remaining -= length
            approx_strategy.append((compound, length))

        total_time = simulate_strategy(
            circuit, approx_strategy, deg_profiles, circuit_info[circuit]['pit_loss']
        )
        if total_time is not None:
            plan_b = {'strategy': approx_strategy, 'total_time': total_time}

    '''Fallback - if no real world plan could be built, use the
    best strategy from the stop count NOT used by Plan A or C'''
    if plan_b is None:
        plan_b = plan_c if plan_c is not None else plan_a

    def format_plan(plan, label):
        if plan is None:
            return None
        risk = calculate_risk_score(
            circuit, plan['strategy'], deg_profiles, hazard_lookup, sc_probability
        )
        return {
            'PlanLabel': label,
            'NumStops': len(plan['strategy']) - 1,
            'strategy': "->".join([f"{c}({l})" for c, l in plan['strategy']]),
            'PredictedTotalTime': round(plan['total_time'], 2),
            'RiskScore': risk,
            'RiskLevel': 'Low' if risk<14 else 'Medium' if risk<20 else 'High'
        }
    return{
        'CircuitName': circuit,
        'PlanA': format_plan(plan_a, 'A'),
        'PlanB': format_plan(plan_b, 'B'),
        'PlanC': format_plan(plan_c, 'C')
    }

MAIN PIPELINE

In [12]:
def main():
    log.info("BOXBOX - Phase 5: Strategy Optimization Engine")
    log.info('-' * 50)

    degradation_df, feature_store, hazard_df, engineered_laps = load_all_data()
    deg_lookup, circuit_info, hazard_lookup = build_lookups(
        degradation_df, feature_store, hazard_df, engineered_laps
    )

    all_plans = []
    circuits = sorted(circuit_info.keys())

    log.info(f"\nGenerating strategies for {len(circuits)} circuits...")

    for circuit in circuits:
        result = build_plans(circuit, deg_lookup, circuit_info, hazard_lookup, engineered_laps)

        if result is None:
            log.warning(f" {circuit}: No valid strategies found")
            continue
        for plan_key in ['PlanA', 'PlanB','PlanC']:
            plan = result[plan_key]
            if plan is None:
                continue
            row = {'CircuitName': circuit}
            row.update(plan)
            all_plans.append(row)

        log.info(f" {circuit}: Plan A = {result['PlanA']['strategy']} "
                f" ({result['PlanA']['PredictedTotalTime']}s, "
                f"risk={result['PlanA']['RiskScore']}")

    plans_df = pd.DataFrame(all_plans)

    #Saving outputs
    plans_path = os.path.join(BASE, 'data', 'processed', 'plan_abc.csv')
    plans_df.to_csv(plans_path, index=False)
    log.info(f"\nSaved: {plans_path} ({len(plans_df)} rows)")

    #Also save just Plan A per circuit as the "optimal_strategies" summary
    optimal_df = plans_df[plans_df['PlanLabel'] == 'A'].reset_index(drop=True)
    optimal_path = os.path.join(BASE, 'data', 'outputs', 'optimal_strategies.csv')
    optimal_df.to_csv(optimal_path, index=False)
    log.info(f" Saved: {optimal_path} ({len(optimal_df)} rows)")

    log.info("\n" + "-" * 50)
    log.info(" Phase 5 Summary")
    log.info("-" * 50)
    log.info(f"\nCircuits with full plan A/B/C: "
            f"{plans_df['CircuitName'].nunique()}")
    log.info(f"\nRisk level breakdown (Plan a only):")
    log.info(optimal_df['RiskLevel'].value_counts().to_string())
    log.info(f"\nStop count breakdown(Plan A only:)")
    log.info(optimal_df['NumStops'].value_counts().to_string())


if __name__ == '__main__':
    main()

2026-07-28 07:20:58,794 - INFO - BOXBOX - Phase 5: Strategy Optimization Engine
2026-07-28 07:20:58,795 - INFO - --------------------------------------------------
2026-07-28 07:20:58,796 - INFO - Loading Phase 3 & 4 outputs...
2026-07-28 07:20:58,896 - INFO -  degradation_profiles: 59 rows
2026-07-28 07:20:58,897 - INFO -  feature_store: 23 rows
2026-07-28 07:20:58,898 - INFO -  cox_hazard_ratios: 69 rows
2026-07-28 07:20:58,899 - INFO - engineered_laps: 20887 rows
2026-07-28 07:20:58,900 - INFO - BUILDING LOOKUP STRCUTURES...
2026-07-28 07:20:58,908 - INFO -  Circuits with degradation data: 23
2026-07-28 07:20:58,909 - INFO -  Circuits with circuit info: 23
2026-07-28 07:20:58,909 - INFO -  Circuits with hazard data: 23
2026-07-28 07:20:58,910 - INFO - 
Generating strategies for 23 circuits...
2026-07-28 07:20:58,944 - INFO -  Abu Dhabi: Plan A = HARD(29)->MEDIUM(29)  (5220.21s, risk=23.7
2026-07-28 07:20:58,962 - INFO -  Australia: Plan A = HARD(19)->MEDIUM(19)->MEDIUM(19)  (4755.22